#### Week 9 - Generative AI: design variations of the same facades

Last week (Week 8) we **recognized and mapped** the buildings. This week we **generate new
variations** of the very same facades - closing the arc *organize -> label -> recognize ->
generate*.

Flow: recap last week (pick a typology + a seed image) -> a one-minute concept ->
Text-to-Render -> Image-to-Render -> curate a variation board. Two ways throughout: the
**xFigura** web platform (no code) and **in-notebook** generation (diffusers, SD-Turbo).

##### > Same data as last week

This uses the same `CASES` as Weeks 7-8. We reuse **your labeled typologies**: pick one,
take a representative facade as the **seed image**, and generate variations from it. To use
your own data, add its folder to `CASES` (as before). Generation needs the **GPU runtime**
(Runtime -> Change runtime type -> T4 GPU).

In [ ]:
# --- one-time installs (Colab); does nothing if already present ---
import importlib.util, subprocess, sys
for pkg, pip_name in [('open_clip', 'open_clip_torch'), ('ipywidgets', 'ipywidgets'),
                      ('minisom', 'minisom')]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pip_name])

import csv, math, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import ipywidgets as widgets
from IPython.display import display
%matplotlib inline

# In Colab, mount Drive and point the CASES paths at your shared folder:
# from google.colab import drive; drive.mount('/content/drive')

# ============ PREPARED CASES (same as Weeks 7-8) ==================
CASES = {
    'detroit': 'data_MLA',           # Week-7 case; coordinates are in the FILENAME
    'gainesville': 'data_gainesville',   # UF street photos; coordinates are in EXIF GPS
}
# ===================================================================

MAX_IMAGES = 400        # cap per perspective (speed / memory safety belt)
SEED = 46
IMG_EXTS = {'.jpg', '.jpeg', '.png'}

def list_images(folder):
    return sorted(p for p in Path(folder).glob('*')
                  if p.is_file() and p.suffix.lower() in IMG_EXTS)

case = widgets.Dropdown(options=list(CASES), value='data_MLA', description='Case:')
display(case)

#### 0. Pick - a case and a perspective

In [ ]:
# PICK: which case + perspective (subfolder) to work on.
CASE_ROOT = Path(CASES[case.value])
assert CASE_ROOT.exists(), f'Folder not found: {CASE_ROOT.resolve()} -- fix its path in CASES.'
PERSPECTIVES = [d.name for d in sorted(CASE_ROOT.iterdir()) if d.is_dir() and list_images(d)]
assert PERSPECTIVES, f'No image subfolders found under {CASE_ROOT}.'

perspective = widgets.Dropdown(options=PERSPECTIVES, value=PERSPECTIVES[0], description='Perspective:')
print('Case:', case.value, '| perspectives:', PERSPECTIVES)
display(perspective)

#### Recap - your typologies from last week (with a backup)

We reuse last week's **labels**. This loads your `labels.csv` if it is there; if not, it
**rebuilds** the typologies from scratch (CLIP -> SOM -> group) so you are never stuck.
Either way you leave with a typology label per image.

In [ ]:
# Get last week's labels for this perspective (load labels.csv, else rebuild).
files = list_images(CASE_ROOT / perspective.value)[:MAX_IMAGES]
assert files, 'No images in this perspective.'
label_csv = CASE_ROOT / perspective.value / 'labels.csv'
y = None
if label_csv.exists():
    by_name = {r['filename']: r['label'] for r in csv.DictReader(open(label_csv, encoding='utf-8'))}
    got = [by_name.get(Path(p).name) for p in files]
    if all(g is not None for g in got):
        y = np.array(got); print('loaded labels from', label_csv)

if y is None:
    print('no labels.csv -- rebuilding typologies from scratch (backup)')
    import torch, open_clip
    from minisom import MiniSom
    from sklearn.cluster import KMeans
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    mdl, _, prep = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
    mdl = mdl.to(dev).eval()
    feats = []
    for i in range(0, len(files), 64):
        ims = [prep(Image.open(p).convert('RGB')) for p in files[i:i + 64]]
        with torch.no_grad():
            f = mdl.encode_image(torch.stack(ims).to(dev)); f = f / f.norm(dim=-1, keepdim=True)
        feats.append(f.cpu().numpy())
    X = np.concatenate(feats).astype(np.float32)
    m = max(3, min(8, int(round((len(X) / 6) ** 0.5))))
    som = MiniSom(m, m, X.shape[1], sigma=1.0, learning_rate=0.5,
                  activation_distance='cosine', random_seed=SEED)
    som.random_weights_init(X); som.train_random(X, 1500)
    win = np.array([som.winner(x) for x in X])
    Wflat = som.get_weights().reshape(m * m, X.shape[1])
    K = max(2, min(6, m))
    grp = KMeans(K, n_init=10, random_state=SEED).fit_predict(Wflat)
    clust = grp[win[:, 0] * m + win[:, 1]]
    y = np.array([f'type_{c}' for c in clust])

classes, counts = np.unique(y, return_counts=True)
print('typologies:', dict(zip(classes.tolist(), counts.tolist())))

#### 1. Pick a typology to redesign, and a seed image

In [ ]:
# PICK a typology (last week's classes) to generate variations for.
typologies = sorted(set(y.tolist()))
typ = widgets.Dropdown(options=typologies, value=typologies[0], description='Typology:')
display(typ)

In [ ]:
# Show that typology's images; pick one as the SEED for image-to-image generation.
members = [p for p, lab in zip(files, y) if lab == typ.value]
print(typ.value, '->', len(members), 'images')
g = min(len(members), 8)
plt.figure(figsize=(2 * g, 2.2))
for k in range(g):
    plt.subplot(1, g, k + 1); plt.imshow(Image.open(members[k]).convert('RGB'))
    plt.axis('off'); plt.title(str(k), fontsize=8)
plt.suptitle(f'{typ.value} - pick a seed index below', fontsize=9); plt.show()

seed = widgets.IntSlider(value=0, min=0, max=max(0, len(members) - 1), description='seed img:')
display(seed)

#### A one-minute concept

A diffusion model turns a **text prompt** (and optionally a **seed image**) into a picture.
Three knobs matter to you as a designer:
- **prompt** - what you ask for (we default it to your typology name),
- **strength** - for image->image, how far to move from your real facade (low = subtle),
- **seed** - the random start; change it for a different variation of the same idea.

Generation is **off** by default (next cell). Turn it on when you are ready.

In [ ]:
# SAFETY BELT: generation is off until you flip this. First 'True' run downloads SD-Turbo.
RUN_GENERATION = False

def _load_pipes():
    # lazy-load SD-Turbo once; text2img + img2img share the same weights (saves memory)
    if globals().get('_pipes'):
        return _pipes
    import torch
    for pk in ['diffusers', 'accelerate']:
        if importlib.util.find_spec(pk) is None:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pk])
    from diffusers import AutoPipelineForText2Image, AutoPipelineForImage2Image
    t2i = AutoPipelineForText2Image.from_pretrained(
        'stabilityai/sd-turbo', torch_dtype=torch.float16).to('cuda')
    i2i = AutoPipelineForImage2Image.from_pipe(t2i)
    globals()['_pipes'] = {'t2i': t2i, 'i2i': i2i}
    print('SD-Turbo ready')
    return _pipes

In [ ]:
# TUNE the generation, and AUTHOR your prompt (default = the typology name).
default_prompt = f'a contemporary {typ.value.replace("_", " ")} building facade, architectural photo'
prompt = widgets.Text(value=default_prompt, description='prompt:', layout=widgets.Layout(width='90%'))
steps = widgets.IntSlider(value=2, min=1, max=4, step=1, description='steps:')
strength = widgets.FloatSlider(value=0.6, min=0.3, max=0.9, step=0.05, description='strength:')
seedno = widgets.IntSlider(value=42, min=0, max=9999, step=1, description='rand seed:')
nvar = widgets.IntSlider(value=4, min=1, max=6, step=1, description='# variations:')
display(widgets.VBox([prompt, steps, strength, seedno, nvar]))

### Section 1 - Text-to-Render (prompt -> image)

The simplest generative task: describe a facade in words, get an image. Your prompt is
pre-filled with the **typology name** from last week, so you are already designing within
the language you discovered.

**Tier A - xFigura (no code):** open **https://xfigura.ai/**, use **Text to Render**, paste
your prompt, download the result.
**Tier B - in the notebook:** run the next cell (SD-Turbo).

In [ ]:
# TIER B - generate in the notebook: text -> image with SD-Turbo.
if RUN_GENERATION:
    import torch
    pipes = _load_pipes()
    gen = torch.Generator('cuda').manual_seed(int(seedno.value))
    img = pipes['t2i'](prompt.value, num_inference_steps=int(steps.value),
                       guidance_scale=0.0, height=512, width=512, generator=gen).images[0]
    plt.figure(figsize=(4, 4)); plt.imshow(img); plt.axis('off')
    plt.title(f'text -> image: "{prompt.value}"', fontsize=8); plt.show()
else:
    print('RUN_GENERATION is off -- flip it in the config cell to generate.')

### Section 2 - Image-to-Render (your facade -> variations)

Now recycle a **real facade** as the seed: the model keeps its structure but restyles it,
controlled by **strength** (low = subtle change, high = bold). This is where last week's
material literally becomes this week's input.

**Tier A - xFigura:** use **Image to Render** (or **Sketch to Render**), upload your seed
facade, add the prompt. **Tier B - in the notebook:** the next cell (SD-Turbo img2img).

In [ ]:
# TIER B - image -> image: keep the building, change the design.
seed_img = Image.open(members[seed.value]).convert('RGB').resize((512, 512))
if RUN_GENERATION:
    import torch
    pipes = _load_pipes()
    gen = torch.Generator('cuda').manual_seed(int(seedno.value))
    var = pipes['i2i'](prompt.value, image=seed_img, strength=float(strength.value),
                       num_inference_steps=max(2, int(steps.value)),
                       guidance_scale=0.0, generator=gen).images[0]
    fig, ax = plt.subplots(1, 2, figsize=(8, 4))
    ax[0].imshow(seed_img); ax[0].set_title('your facade (seed)'); ax[0].axis('off')
    ax[1].imshow(var); ax[1].set_title(f'variation (strength {strength.value})'); ax[1].axis('off')
    plt.tight_layout(); plt.show()
else:
    plt.figure(figsize=(4, 4)); plt.imshow(seed_img); plt.axis('off')
    plt.title('seed facade (RUN_GENERATION is off)'); plt.show()

### Section 3 - Curate a variation board

Generate several variations (same prompt + seed image, different random seeds) and lay them
out as a board - your design-ideation deliverable. The last cell displays the renders you
made in **xFigura** (Tier A), so both paths land in the same place.

In [ ]:
# Build a board of variations (same seed image + prompt, different random seeds) and save it.
if RUN_GENERATION:
    import torch
    pipes = _load_pipes()
    base = Image.open(members[seed.value]).convert('RGB').resize((512, 512))
    outdir = Path('generated_board'); outdir.mkdir(exist_ok=True)
    N = int(nvar.value); cols = min(N, 3); rows = math.ceil(N / cols)
    plt.figure(figsize=(3 * cols, 3 * rows))
    for k in range(N):
        gen = torch.Generator('cuda').manual_seed(int(seedno.value) + k)
        v = pipes['i2i'](prompt.value, image=base, strength=float(strength.value),
                         num_inference_steps=max(2, int(steps.value)),
                         guidance_scale=0.0, generator=gen).images[0]
        v.save(outdir / f'var_{k}.png')
        plt.subplot(rows, cols, k + 1); plt.imshow(v); plt.axis('off')
        plt.title(f'seed {int(seedno.value) + k}', fontsize=8)
    plt.suptitle(f'variation board - {typ.value}', fontsize=10)
    plt.tight_layout(); plt.show()
    print('saved', N, 'images ->', outdir)
else:
    print('RUN_GENERATION is off -- flip it in the config cell to build the board.')

In [ ]:
# (Tier A) show the renders you downloaded from xFigura, if any.
xdir = Path('xfigura_renders')          # make this folder and drop your xFigura downloads in it
xf = list_images(xdir) if xdir.exists() else []
if xf:
    cols = min(len(xf), 3); rows = math.ceil(len(xf) / cols)
    plt.figure(figsize=(3 * cols, 3 * rows))
    for k, p in enumerate(xf):
        plt.subplot(rows, cols, k + 1); plt.imshow(Image.open(p).convert('RGB')); plt.axis('off')
    plt.suptitle('xFigura renders', fontsize=10); plt.tight_layout(); plt.show()
else:
    print('No xFigura renders found -- make a folder "xfigura_renders" and drop images in it.')